In [2]:
from elevant.models.entity_database import EntityDatabase
entity_db = EntityDatabase()
entity_db.load_entity_names()
entity_db.load_alias_to_entities()
entity_db.load_hyperlink_to_most_popular_candidates()
entity_db.load_sitelink_counts()

In [17]:
import os, json
with open('/media/volume/LLMRag2/.local/doid-results/llmner_graph_llm_doid.jsonl', 'r') as f:
    for line in f:
        data = json.loads(line)
        for mention in data.get('entity_mentions', []):
            spans = mention['span']
            candidate = mention['candidates']
            candidate_dict = {i['id']: i for i in candidate}
            print(data['text'][spans[0]:spans[1]], '->', candidate_dict.get(mention['id'], {'name': '<unknown>'})['name'])


agranulocytosis -> <unknown>
agranulocytic angina -> <unknown>
septicaemia -> <unknown>
dyspnoea -> dyspnea
scarlet fever -> scarlet fever
scarlet fever -> scarlet fever
septicaemia -> <unknown>
cervical cellulitis -> cellulitis
myocarditis -> <unknown>
toxic scarlet fever -> scarlet fever
measles -> <unknown>
bronchopneumonia -> <unknown>
bronchitis -> bronchitis
subacute bacterial endocarditis -> subacute bacterial endocarditis
Streptococcus viridans infectio -> <unknown>
Streptococcus viridans infection -> obsolete commensal streptococcal infectious disease
subacute bacterial group -> <unknown>
cretinism -> congenital hypothyroidism
myxocdema -> <unknown>
sporadic cretinism -> <unknown>
cachexia strumpirva -> <unknown>
endemic cretinism -> <unknown>
cretinism -> congenital hypothyroidism
cretinism -> congenital hypothyroidism
myxoedema -> <unknown>
sporadic cretinism -> congenital hypothyroidism
endemic cretinism -> <unknown>
varicose ulcers -> <unknown>
eczema -> <unknown>
s -> nos

In [1]:
from elevant.linkers.graph_linker import GraphLinker, OBOEntityLinker

linker = OBOEntityLinker(obo_path='/media/volume/LLMRag2/.local/obo/human_genes.obo')


/media/volume/LLMRag2/miniconda3/envs/running/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/media/volume/LLMRag2/miniconda3/envs/running/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loaded: 64488163 entities, 102872286 names (OBO file)
Database: /media/volume/LLMRag2/.local/obo_cache/human_genes_359916ed.db


In [1]:
import sqlite3
import re
from pathlib import Path
from typing import Dict, Tuple, Set
import functools
import nltk
from rapidfuzz import fuzz
STOP = set(nltk.corpus.stopwords.words('english'))


class OBOEntityLinker:
    """Link entities tối ưu: id() ~0.001ms, link() vẫn dùng FTS5."""
    
    def __init__(self, obo_path: str):
        self.conn = sqlite3.connect(":memory:")
        self.cur = self.conn.cursor()
        
        # ✅ Tạo cả 2 bảng: entities (nhanh) + names_fts (link)
        self.cur.executescript("""
            CREATE TABLE entities (
                id TEXT PRIMARY KEY, 
                name TEXT, 
                def TEXT,
                aliases TEXT  -- Pipe-separated synonyms
            );
            
            CREATE VIRTUAL TABLE names_fts USING fts5(
                entity_id UNINDEXED, 
                name, 
                tokenize='trigram', 
                prefix='2 3'
            );
            
            -- Tối ưu SQLite
            PRAGMA journal_mode=WAL;
            PRAGMA synchronous=NORMAL;
            PRAGMA cache_size=-200000;  -- 200MB RAM cache
        """)
        
        self._load(obo_path)
        
        # ✅ Preload tất cả aliases vào memory cho id()
        self._synonym_cache = self._build_synonym_cache()
    
    def _load(self, path: str):
        term = {'synonyms': []}
        with open(path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line == '[Term]':
                    if 'id' in term: 
                        self._insert(term)
                    term = {'synonyms': []}
                elif m := re.match(r'id: (.+)', line): term['id'] = m.group(1)
                elif m := re.match(r'name: (.+)', line): term['name'] = m.group(1)
                elif m := re.match(r'def: "(.+?)"', line): term['def'] = m.group(1)
                elif m := re.match(r'synonym: "(.+?)"', line): term['synonyms'].append(m.group(1))
        
        if 'id' in term: 
            self._insert(term)
        self.conn.commit()
        self._stats()
    
    def _insert(self, term: Dict):
        all_names = [n for n in [term.get('name')] + term['synonyms'] if n]
        
        # ✅ Lưu aliases vào entities (cho id nhanh)
        aliases_str = '|'.join(all_names[1:]) if len(all_names) > 1 else ''
        self.cur.execute(
            "INSERT INTO entities VALUES (?, ?, ?, ?)", 
            (term['id'], term.get('name'), term.get('def'), aliases_str)
        )
        
        for name in set(all_names):
            if name:
                self.cur.execute(
                    "INSERT INTO names_fts VALUES (?, ?)", 
                    (term['id'], name.lower())
                )
    
    def _stats(self):
        self.cur.execute("SELECT COUNT(*) FROM entities")
        total = self.cur.fetchone()[0]
        print(f"✅ Loaded {total} entities")

    def _build_synonym_cache(self) -> Dict[str, list]:
        cache = {}
        self.cur.execute("SELECT id, aliases FROM entities")
        
        for entity_id, aliases_str in self.cur.fetchall():
            cache[entity_id] = aliases_str.split('|') if aliases_str else []
        
        return cache

    @functools.lru_cache(maxsize=32768)  # ✅ Cache kết quả hàm
    def id(self, entity_id: str) -> Dict:
        if entity_id not in self._synonym_cache:
            return {'error': 'Entity not found'}
        
        # Chỉ query 1 lần lấy name/def
        self.cur.execute(
            "SELECT name, def FROM entities WHERE id = ?", 
            (entity_id,)
        )
        row = self.cur.fetchone()
        
        if not row: 
            return {'error': 'Entity not found'}
        
        name, definition = row
        
        return {
            'id': entity_id,
            'label': name,
            'description': definition or '',
            'aliases': self._synonym_cache[entity_id]  # Lấy từ memory
        }

    def link(self, text: str, thr: int = 85, k: int = 10, max_stopword_ratio: float = 0.5) -> Dict[Tuple[int, int], Dict]:
        text = text.lower()
        words = [(m.start(), m.end(), m.group()) for m in re.finditer(r'\S+', text)]
        
        spans = [(words[i][0], words[i+n-1][1], ' '.join(w[2] for w in words[i:i+n]))
                for n in range(1, min(5, len(words)+1))
                for i in range(len(words)-n+1)
                if (sum(1 for w in words[i:i+n] if w[2].lower() in STOP) / n) <= max_stopword_ratio]
        
        results = {}
        for start, end, span in spans:
            clean_span = re.sub(r'[.:*^$+-]', ' ', span.lower()).strip()
            
            # Query names_fts như cũ
            self.cur.execute(f"""
                SELECT DISTINCT e.id, e.name, e.def, n.name 
                FROM names_fts n
                JOIN entities e ON n.entity_id = e.id
                WHERE n.name MATCH ? 
                ORDER BY rank
                LIMIT {k}
            """, (f'"{clean_span}"',))
            
            cands = self.cur.fetchall()
            
            if not cands and ' ' in span:
                for w in span.split():
                    if w.lower() in STOP: continue
                    clean_word = re.sub(r'[.:*^$+-]', ' ', w.lower()).strip()
                    self.cur.execute(f"""
                        SELECT e.id, e.name, e.def, n.name FROM names_fts n
                        JOIN entities e ON n.entity_id = e.id
                        WHERE n.name MATCH ? LIMIT {k}
                    """, (f'"{clean_word.replace('"', '').replace("'", '').strip()}"',))
                    cands.extend(self.cur.fetchall())
            
            best = {}
            for qid, name, definition, matched in cands:
                score = fuzz.WRatio(span, matched)
                if score >= thr and (qid not in best or score > best[qid]['score']):
                    best[qid] = {
                        'id': qid, 
                        'name': name, 
                        'def': definition, 
                        'matched_term': matched, 
                        'score': score
                    }
            
            if best:
                results[(start, end)] = {
                    'span_text': span,
                    'entities': sorted(best.values(), key=lambda x: x['score'], reverse=True)[:k]
                }
        
        return results

    def close(self):
        """Clear cache khi đóng."""
        self.id.cache_clear()
        self._synonym_cache.clear()
        if hasattr(self, 'conn'):
            self.conn.close()


import time

linker = OBOEntityLinker('/media/volume/LLMRag2/.local/obo/human_genes.obo')

start = time.perf_counter()
for _ in range(1000):
    result = linker.id('NCBIGene:97918332')
print(f"⏱️  id() x 1000: {(time.perf_counter()-start)*1000:.2f}ms")

start = time.perf_counter()
results = linker.link("This is a test p53 sequence")
print(f"⏱️  link() x 1: {(time.perf_counter()-start)*1000:.2f}ms")

✅ Loaded 64488163 entities
⏱️  id() x 1000: 0.37ms
⏱️  link() x 1: 60.71ms


In [18]:
linker = OBOEntityLinker(obo_path='/media/volume/LLMRag2/.local/obo/human_genes.obo')
linker.id('NCBIGene:97918332')

✅ Loaded 64488163 entities from OBO


{'id': 'NCBIGene:97918332',
 'label': 'H4K01_RS00045',
 'description': 'hypothetical protein',
 'aliases': ['D728_p11', 'H4K01_RS00045']}

In [23]:
# import nltk
# STOP = set(nltk.corpus.stopwords.words('english'))

linker.id('NCBIGene:97918330')

{'id': 'NCBIGene:97918330',
 'label': 'H4K01_RS00035',
 'description': 'hypothetical protein',
 'aliases': ['D728_p09', 'H4K01_RS00035']}

In [ ]:
# linker.id('NCBIGene:97918332')

linker.cur.execute(f"SELECT name, def FROM entities WHERE id = ?", ('NCBIGene:97918332',))
row = linker.cur.fetchone()

linker.cur.execute(f"SELECT name FROM names_fts WHERE entity_id = ? LIMIT 10", ('NCBIGene:97918332',))
out = linker.cur.fetchall()
# all_names = [r[0] for r in linker.cur.fetchall()]
# all_names
out
# linker.link('p53')

In [ ]:
linker._mrel(a)

In [1]:
import json
import sys

def convert_bc2gn_to_jsonl(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Split into blocks separated by double newlines
    blocks = [block.strip() for block in content.strip().split('\n\n') if block.strip()]
    
    results = []
    
    for block_idx, block in enumerate(blocks):
        lines = [line.strip() for line in block.split('\n') if line.strip()]
        
        title_text = ""
        abstract_text = ""
        pmid = ""
        labels = []
        
        for line in lines:
            # Parse title line
            if '|t|' in line:
                parts = line.split('|', 2)
                if len(parts) >= 3:
                    pmid = parts[0]
                    title_text = parts[2].strip()
            
            # Parse abstract line
            elif '|a|' in line:
                parts = line.split('|', 2)
                if len(parts) >= 3:
                    abstract_text = parts[2].strip()
            
            # Parse annotation lines (tab-separated)
            else:
                parts = line.split('\t')
                if len(parts) >= 5:
                    # Extract annotation fields
                    start = int(parts[1])
                    end = int(parts[2])
                    text = parts[3]
                    ent_type = parts[4]
                    
                    # Get entity_id if present (skip if missing or "-")
                    entity_id = parts[5] if len(parts) >= 6 and parts[5] and parts[5] != "-" else None
                    
                    # Only include annotations with valid entity_id
                    if entity_id:
                        label = {
                            "id": len(labels),  # Sequential ID within document
                            "span": [start, end],
                            "entity_id": entity_id,
                            "name": text,
                            "type": ent_type
                        }
                        labels.append(label)
        
        # Concatenate title and abstract with a space separator
        full_text = f"{title_text} {abstract_text}".strip()
        
        # Skip empty documents
        if not full_text:
            continue
        
        # Create document JSON object
        doc = {
            "id": block_idx,  # Sequential document ID starting from 0
            "title": pmid,    # Use PMID as title identifier
            "text": full_text,
            "evaluation_span": [0, len(full_text)],
            "labels": labels
        }
        results.append(doc)
    
    # Write to JSONL file (one JSON per line)
    with open(output_file, 'w', encoding='utf-8') as f:
        for doc in results:
            json.dump(doc, f, ensure_ascii=False)
            f.write('\n')
    
    print(f"Successfully converted {len(results)} documents to {output_file}")

convert_bc2gn_to_jsonl('/media/volume/LLMRag2/.local/ActDiseaseEL/BC2GNtest.PubTator.txt', './BC2GNtest.PubTator.jsonl')

Successfully converted 262 documents to ./BC2GNtest.PubTator.jsonl


In [ ]:
import spacy

nlp = spacy.load('en_ner_bc5cdr_md')

'''
en_ner_jnlpba_md: genes
en_ner_bc5cdr_md: diseases + chemicals
en_core_web_lg: general, wiki
'''

text = '''ICSBP is essential for the development of mouse type I interferon-producing cells and for the generation and activation of CD8alpha(+) dendritic cells. Interferon (IFN) consensus sequence-binding protein (ICSBP) is a transcription factor playing a critical role in the regulation of lineage commitment, especially in myeloid cell differentiation. In this study, we have characterized the phenotype and activation pattern of subsets of dendritic cells (DCs) in ICSBP(-/-) mice. Remarkably, the recently identified mouse IFN-producing cells (mIPCs) were absent in all lymphoid organs from ICSBP(-/-) mice, as revealed by lack of CD11c(low)B220(+)Ly6C(+)CD11b(-) cells. In parallel, CD11c(+) cells isolated from ICSBP(-/-) spleens were unable to produce type I IFNs in response to viral stimulation. ICSBP(-/-) mice also displayed a marked reduction of the DC subset expressing the CD8alpha marker (CD8alpha(+) DCs) in spleen, lymph nodes, and thymus. Moreover, ICSBP(-/-) CD8alpha(+) DCs exhibited a markedly impaired phenotype when compared with WT DCs. They expressed very low levels of costimulatory molecules (intercellular adhesion molecule [ICAM]-1, CD40, CD80, CD86) and of the T cell area-homing chemokine receptor CCR7, whereas they showed higher levels of CCR2 and CCR6, as revealed by reverse transcription PCR. In addition, these cells were unable to undergo full phenotypic activation upon in vitro culture in presence of maturation stimuli such as lipopolysaccharide or poly (I:C), which paralleled with lack of Toll-like receptor (TLR)3 mRNA expression. Finally, cytokine expression pattern was also altered in ICSBP(-/-) DCs, as they did not express interleukin (IL)-12p40 or IL-15, but they displayed detectable IL-4 mRNA levels. On the whole, these results indicate that ICSBP is a crucial factor in the regulation of two possibly linked processes: (a) the development and activity of mIPCs, whose lack in ICSBP(-/-) mice may explain their high susceptibility to virus infections; (b) the generation and activation of CD8alpha(+) DCs, whose impairment in ICSBP(-/-) mice can be responsible for the defective generation of a Th1 type of immune response.'''

doc = nlp(text)

print("=== BC5CDR NER ===")
for ent in doc.ents:
    print(ent.span)
    print(f"- {ent.text:<15} | Loại: {ent.label_:<10}")

=== BC5CDR NER ===
lipopolysaccharide
- lipopolysaccharide | Loại: CHEMICAL  
poly
- poly            | Loại: CHEMICAL  
infections
- infections      | Loại: DISEASE   


In [2]:
from elevant.linkers.graph_linker import add_correct_id_to_entity_ids

# D001943
add_correct_id_to_entity_ids({
    'start_pos': 996,
    'end_pos': 1015,
})

/media/volume/LLMRag2/miniconda3/envs/running/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/media/volume/LLMRag2/miniconda3/envs/running/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


['MESH:D001943']

In [5]:
from typing import Dict, Tuple
import sqlite3, re, rapidfuzz.fuzz as fuzz, nltk
import functools
STOP = set(nltk.corpus.stopwords.words('english'))

class OBOEntityLinker:
    def __init__(self, obo_path: str):
        self.conn = sqlite3.connect(":memory:")
        self.cur = self.conn.cursor()
        
        self.cur.executescript("""
            CREATE TABLE entities (
                id TEXT PRIMARY KEY, 
                name TEXT, 
                def TEXT,
                aliases TEXT  -- Pipe-separated synonyms
            );
            
            CREATE VIRTUAL TABLE names_fts USING fts5(
                entity_id UNINDEXED, 
                name, 
                tokenize='trigram', 
                prefix='2 3'
            );
            
            -- Tối ưu SQLite
            PRAGMA journal_mode=WAL;
            PRAGMA synchronous=NORMAL;
            PRAGMA cache_size=-200000;  -- 200MB RAM cache
        """)
        
        self._load(obo_path)
        
        self._synonym_cache = self._build_synonym_cache()
    
    def _load(self, path: str):
        term = {'synonyms': []}
        with open(path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line == '[Term]':
                    if 'id' in term: 
                        self._insert(term)
                    term = {'synonyms': []}
                elif m := re.match(r'id: (.+)', line): term['id'] = m.group(1)
                elif m := re.match(r'name: (.+)', line): term['name'] = m.group(1)
                elif m := re.match(r'def: "(.+?)"', line): term['def'] = m.group(1)
                elif m := re.match(r'synonym: "(.+?)"', line): term['synonyms'].append(m.group(1))
        
        if 'id' in term: 
            self._insert(term)
        self.conn.commit()
        self._stats()
    
    def _insert(self, term: Dict):
        all_names = [n for n in [term.get('name')] + term['synonyms'] if n]
        
        aliases_str = '|'.join(all_names[1:]) if len(all_names) > 1 else ''
        self.cur.execute(
            "INSERT INTO entities VALUES (?, ?, ?, ?)", 
            (term['id'], term.get('name'), term.get('def'), aliases_str)
        )
        
        for name in set(all_names):
            if name:
                self.cur.execute(
                    "INSERT INTO names_fts VALUES (?, ?)", 
                    (term['id'], name.lower())
                )
    
    def _stats(self):
        self.cur.execute("SELECT COUNT(*) FROM entities")
        total = self.cur.fetchone()[0]
        print(f"Total entities: {total}")

    def _build_synonym_cache(self) -> Dict[str, list]:
        cache = {}
        self.cur.execute("SELECT id, aliases FROM entities")
        
        for entity_id, aliases_str in self.cur.fetchall():
            cache[entity_id] = aliases_str.split('|') if aliases_str else []
        
        return cache

    @functools.lru_cache(maxsize=32768)
    def id(self, entity_id: str) -> Dict:
        if entity_id not in self._synonym_cache:
            return {'error': 'Entity not found'}
        
        self.cur.execute(
            "SELECT name, def FROM entities WHERE id = ?", 
            (entity_id,)
        )
        row = self.cur.fetchone()
        
        if not row: 
            return {'error': 'Entity not found'}
        
        name, definition = row
        
        return {
            'id': entity_id,
            'label': name,
            'description': definition or '',
            'aliases': self._synonym_cache[entity_id]
        }

    def link(self, text: str, thr: int = 85, k: int = 10, max_stopword_ratio: float = 0.5) -> Dict[Tuple[int, int], Dict]:
        text = text.lower()
        words = [(m.start(), m.end(), m.group()) for m in re.finditer(r'\S+', text)]
        
        spans = [(words[i][0], words[i+n-1][1], ' '.join(w[2] for w in words[i:i+n]))
                for n in range(1, min(5, len(words)+1))
                for i in range(len(words)-n+1)
                if (sum(1 for w in words[i:i+n] if w[2].lower() in STOP) / n) <= max_stopword_ratio]
        
        results = {}
        for start, end, span in spans:
            clean_span = re.sub(r'[.:*^$+-]', ' ', span.lower()).strip().replace('"', '').replace("'", '').strip()
            
            self.cur.execute(f"""
                SELECT DISTINCT e.id, e.name, e.def, n.name 
                FROM names_fts n
                JOIN entities e ON n.entity_id = e.id
                WHERE n.name MATCH ? 
                ORDER BY rank
                LIMIT {k}
            """, (f'"{clean_span}"',))
            
            cands = self.cur.fetchall()
            
            if not cands and ' ' in span:
                for w in span.split():
                    if w.lower() in STOP: continue
                    clean_word = re.sub(r'[.:*^$+-]', ' ', w.lower()).strip()
                    self.cur.execute(f"""
                        SELECT e.id, e.name, e.def, n.name FROM names_fts n
                        JOIN entities e ON n.entity_id = e.id
                        WHERE n.name MATCH ? LIMIT {k}
                    """, (f'"{clean_word.replace('"', '').replace("'", '').strip()}"',))
                    cands.extend(self.cur.fetchall())
            
            best = {}
            for qid, name, definition, matched in cands:
                score = fuzz.WRatio(span, matched)
                if score >= thr and (qid not in best or score > best[qid]['score']):
                    best[qid] = {
                        'id': qid, 
                        'name': name, 
                        'def': definition, 
                        'matched_term': matched, 
                        'score': score
                    }
            
            if best:
                results[(start, end)] = {
                    'span_text': span,
                    'entities': sorted(best.values(), key=lambda x: x['score'], reverse=True)[:k]
                }
        
        return results

    def close(self):
        """Clear cache khi đóng."""
        self.id.cache_clear()
        self._synonym_cache.clear()
        if hasattr(self, 'conn'):
            self.conn.close()

linker = OBOEntityLinker('/media/volume/LLMRag2/.local/obo/CTD_diseases_filtered.obo')
linker.link('insulin-dependent diabetes mellitus')

Total entities: 9633


{(18, 26): {'span_text': 'diabetes',
  'entities': [{'id': 'MESH:D003921',
    'name': 'Diabetes Mellitus, Experimental',
    'def': 'Diabetes mellitus induced experimentally by administration of various diabetogenic agents or by PANCREATECTOMY.',
    'matched_term': 'alloxan diabetes',
    'score': 90.0},
   {'id': 'OMIM:616087',
    'name': 'TYPE 2 DIABETES 5',
    'def': None,
    'matched_term': 'type 2 diabetes 5',
    'score': 90.0},
   {'id': 'MESH:D007015',
    'name': 'Hypophosphatemia, Familial',
    'def': 'An inherited condition of abnormally low serum levels of PHOSPHATES (below 1 mg/liter) which can occur in a number of genetic diseases with defective reabsorption of inorganic phosphorus by the PROXIMAL RENAL TUBULES. This leads to phosphaturia, HYPOPHOSPHATEMIA, and disturbances of cellular and organ functions such as those in X-LINKED HYPOPHOSPHATEMIC RICKETS; OSTEOMALACIA; and FANCONI SYNDROME.',
    'matched_term': 'phosphate diabetes',
    'score': 90.0},
   {'id': '